# Convolución

**Capítulo 4 · Universidad de las Hespérides**

Adaptación al español de *Dive into Deep Learning*, Aston Zhang, Zachary C. Lipton, Mu Li y Alexander J. Smola.
Fuente: `locked/chapter_convolutional-neural-networks/conv-layer.ipynb` · [Lección original](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html).
Texto adaptado bajo [CC BY-SA 4.0](https://creativecommons.org/licenses/by-sa/4.0/). [Procedencia y cambios](../PROCEDENCIA.md).
Se conserva la secuencia de las celdas y de los ejercicios; las notas de Hespérides se identifican expresamente.

**Entorno:** ejecuta `uv sync` en la raíz y selecciona su Python como kernel. Las descargas se realizan una vez y quedan en `data/`.
Por defecto, el soporte limita los entrenamientos de `Trainer` a tres épocas y 1024/256 ejemplos para CPU.
Para repetir el régimen completo, inicia Jupyter con `HESPERIDES_COMPLETO=1`. Los ejemplos visuales pequeños conservan su propia configuración explícita.
Los datos de texto en inglés o francés son entradas de los experimentos originales y mantienen su idioma.


In [ ]:
from pathlib import Path
import sys
RAIZ = Path.cwd() if (Path.cwd() / "laboratorio").exists() else Path.cwd().parent
if str(RAIZ) not in sys.path:
    sys.path.insert(0, str(RAIZ))
from laboratorio import d2l, configurar, epocas
configurar()


# Convoluciones para imágenes
<a id="sec_conv_layer"></a>

Ahora que entendemos cómo funcionan las capas convolucionales en la teoría, estamos listos para ver cómo funcionan en la práctica. Sobre la base de nuestra motivación de las redes neuronales convolucionales como arquitecturas eficientes para explorar la estructura en los datos de imagen, nos quedamos con las imágenes como nuestro ejemplo de trabajo.


In [ ]:
import torch
from torch import nn
from laboratorio import d2l

## La operación de correlación cruzada
Recuerde que estrictamente hablando, las capas convolucionales son un nombre erróneo, ya que las operaciones que expresan se describen con mayor precisión como correlaciones cruzadas. Basados en nuestras descripciones de capas convolucionales en [Referencia sec_why-conv](https://d2l.ai/chapter_convolutional-neural-networks/why-conv.html#sec-why-conv), en tal capa, un tensor de entrada y un tensor de núcleo se combinan para producir un tensor de salida a través de una operación de correlación transversal **.

Ignoremos los canales por ahora y veamos cómo funciona esto con datos bidimensionales y representaciones ocultas. En [Referencia fig_correlation](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html#fig-correlation), la entrada es un tensor bidimensional con una altura de 3 y una anchura de 3. Marcamos la forma del tensor como $3 \times 3$ o ($3$, $3$). La altura y anchura del núcleo son ambas 2. La forma de la ventana *kernel* (o *ventana de convolución*) está dada por la altura y anchura del núcleo (aquí es $2 \times 2$).

![Correlación cruzada bidimensional. Se resaltan la primera salida y los elementos de entrada y filtro que la producen: $0\times0+1\times1+3\times2+4\times3=19$.](../recursos/originales/correlation.svg)
<a id="fig_correlation"></a>

En la operación de correlación cruzada bidimensional, comenzamos con la ventana de convolución situada en la esquina superior izquierda del tensor de entrada y la deslizamos a través del tensor de entrada, tanto de izquierda a derecha como de arriba a abajo. Cuando la ventana de convolución se desliza a cierta posición, el subtensor de entrada contenido en esa ventana y el tensor del núcleo se multiplican en sentido de elemento y el tensor resultante se resume dando un solo valor escalar. Este resultado da el valor del tensor de salida en la ubicación correspondiente. Aquí, el tensor de salida tiene una altura de 2 y anchura de 2 y los cuatro elementos se derivan de la operación de correlación cruzada bidimensional:

$$
0\times0+1\times1+3\times2+4\times3=19,\\
1\times0+2\times1+4\times2+5\times3=25,\\
3\times0+4\times1+6\times2+7\times3=37,\\
4\times0+5\times1+7\times2+8\times3=43.
$$

Tenga en cuenta que a lo largo de cada eje, el tamaño de salida es ligeramente menor que el tamaño de entrada. Debido a que el núcleo tiene anchura y altura mayores que $1$, sólo podemos calcular correctamente la correlación cruzada para las ubicaciones donde el núcleo encaja totalmente dentro de la imagen, el tamaño de salida se da por el tamaño de entrada $n_\textrm{h} \times n_\textrm{w}$ menos el tamaño del núcleo de convolución $k_\textrm{h} \times k_\textrm{w}$ vía

$$(n_\textrm{h}-k_\textrm{h}+1) \times (n_\textrm{w}-k_\textrm{w}+1).$$

Este es el caso ya que necesitamos espacio suficiente para "desplazar" el núcleo de la convolución a través de la imagen. Más adelante veremos cómo mantener el tamaño sin cambios mediante el relleno de la imagen con ceros alrededor de su límite para que haya espacio suficiente para desplazar el núcleo. A continuación, implementamos este proceso en la función `corr2d`, que acepta un tensor de entrada `X` y un tensor de núcleo `K` y devuelve un tensor de salida `Y`.


In [ ]:
def corr2d(X, K):  #@save
    """Calcular la correlación cruzada 2D."""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y

Podemos construir el tensor de entrada `X` y el tensor de núcleo `K` de [Referencia fig_correlation](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html#fig-correlation) a **validar la salida de la implementación anterior** de la operación de correlación cruzada bidimensional.


In [ ]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]])
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]])
corr2d(X, K)

## Capas convolucionales
Una capa convolucional corre transversalmente la entrada y el núcleo y añade un sesgo escalar para producir una salida. Los dos parámetros de una capa convolucional son el núcleo y el sesgo escalar. Cuando se entrenan modelos basados en capas convolucionales, normalmente inicializamos los núcleos al azar, al igual que lo haríamos con una capa totalmente conectada.

Ahora estamos listos para **implementar una capa convolucional bidimensional** basada en la función `corr2d` definida anteriormente. En el método constructor `__init__`, declaramos `weight` y `bias` como los dos parámetros del modelo. El método de propagación hacia delante llama a la función `corr2d` y añade el sesgo.


In [ ]:
class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))

    def forward(self, x):
        return corr2d(x, self.weight) + self.bias

En $h \times w$ convolution o un núcleo de convolution $h \times w$, la altura y anchura del núcleo de convolution son $h$ y $w$, respectivamente. También nos referimos a una capa convolutional con un núcleo de convolution $h \times w$ simplemente como una capa convolutional $h \times w$.

## Detección de bordes de objetos en imágenes
Tomemos un momento para analizar ** una simple aplicación de una capa convolucional: detectando el borde de un objeto en una imagen** encontrando la ubicación del cambio de píxel. Primero, construimos una "imagen" de píxeles $6\times 8$. Las cuatro columnas centrales son negras ($0$) y el resto son blancas ($1$).


In [ ]:
X = torch.ones((6, 8))
X[:, 2:6] = 0
X

A continuación, construimos un núcleo `K` con una altura de 1 y una anchura de 2. Cuando realizamos la operación de correlación cruzada con la entrada, si los elementos adyacentes horizontales son los mismos, la salida es 0. De lo contrario, la salida es no cero. Tenga en cuenta que este núcleo es un caso especial de un operador de diferencia finita. En la ubicación $(i,j)$ calcula $x_{i,j} - x_{(i+1),j}$, es decir, calcula la diferencia entre los valores de píxeles adyacentes horizontales. Esta es una aproximación discreta de la primera derivada en la dirección horizontal. Después de todo, para una función $f(i,j)$ su derivada $-\partial_i f(i,j) = \lim_{\epsilon \to 0} \frac{f(i,j) - f(i+\epsilon,j)}{\epsilon}$. Veamos cómo funciona en la práctica.


### Nota docente de Hespérides

Una capa convolucional reutiliza el mismo filtro en posiciones distintas: comparte parámetros y explota localidad. Para una dimensión, la salida tiene tamaño $\lfloor(n+2p-d(k-1)-1)/s+1\rfloor$, con padding $p$, dilatación $d$ y stride $s$. El campo receptivo crece al componer capas, aunque cada filtro sea pequeño. En el explorador 90 puedes seguir la ventana y comprobar que la suma de productos coincide con la celda correspondiente de la salida.

Vínculo con los apuntes: sesión 4, «Convolución».


In [ ]:
K = torch.tensor([[1.0, -1.0]])

Estamos listos para realizar la operación de correlación cruzada con argumentos `X` (nuestra entrada) y `K` (nuestro núcleo). Como puede ver, ** detectamos $1$ para el borde de blanco a negro y $-1$ para el borde de negro a blanco.** Todas las demás salidas toman valor $0$.


In [ ]:
Y = corr2d(X, K)
Y

Ahora podemos aplicar el núcleo a la imagen transpuesta. Como se esperaba, desaparece. **El núcleo `K` solo detecta bordes verticales.**


In [ ]:
corr2d(X.t(), K)

## Aprender un kernel
Diseñar un detector de bordes por diferencias finitas `[1, -1]` es limpio si sabemos que esto es precisamente lo que estamos buscando. Sin embargo, al mirar a núcleos más grandes, y considerar capas sucesivas de convoluciones, podría ser imposible especificar con precisión lo que cada filtro debe estar haciendo manualmente.

Ahora veamos si podemos **aprender el núcleo que generó `Y` de `X`** mirando los pares de entrada--salida solamente. Primero construimos una capa convolucional e inicializamos su núcleo como un tensor aleatorio. Luego, en cada iteración, usaremos el error cuadrado para comparar `Y` con la salida de la capa convolucional. Luego podemos calcular el gradiente para actualizar el núcleo. Por el bien de la simplicidad, en el siguiente usamos la clase integrada para las capas convolucionales bidimensionales e ignorar el sesgo.


In [ ]:
# Construir una capa convolucional bidimensional con 1 canal de salida y un
# núcleo de la forma (1, 2). Por el bien de la simplicidad, ignoramos el sesgo aquí
conv2d = nn.LazyConv2d(1, kernel_size=(1, 2), bias=False)

# La capa convolucional bidimensional utiliza la entrada de cuatro dimensiones y
# salida en el formato de (ejemplo, canal, altura, anchura), donde el lote
# tamaño (número de ejemplos en el lote) y el número de canales son ambos 1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2  # Tasa de aprendizaje

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    # Actualizar el núcleo
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i + 1}, loss {l.sum():.3f}')

Tenga en cuenta que el error ha caído a un pequeño valor después de 10 iteraciones. Ahora vamos a ** echar un vistazo al tensor del núcleo que aprendimos.**


In [ ]:
conv2d.weight.data.reshape((1, 2))

De hecho, el tensor del núcleo aprendido está notablemente cerca del tensor del núcleo `K` que definimos anteriormente.

## Correlación cruzada y convolución
Recordemos nuestra observación desde [Referencia sec_why-conv](https://d2l.ai/chapter_convolutional-neural-networks/why-conv.html#sec-why-conv) de la correspondencia entre la correlación cruzada y las operaciones de convolución. Aquí vamos a seguir considerando las capas convolucionales bidimensionales. ¿Qué pasa si tales capas realizan operaciones de convolución estrictas como se define en [Referencia eq_2d-conv-discrete](https://d2l.ai/#eq-2d-conv-discrete) en lugar de las correlaciones cruzadas? Para obtener la salida de la operación estricta *convolución*, sólo necesitamos voltear el tensor del núcleo bidimensional tanto horizontal como verticalmente, y luego realizar la operación *correlaciones cruzadas* con el tensor de entrada.

Cabe señalar que, dado que los núcleos se aprenden de los datos en el aprendizaje profundo, las salidas de las capas convolucionales no se ven afectadas, independientemente de que dichas capas realicen las operaciones de convolución estrictas o las operaciones de correlación cruzada.

Para ilustrar esto, supongamos que una capa convolucional realiza *correlación cruzada* y aprende el núcleo en [Referencia fig_correlation](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html#fig-correlation), que aquí se denota como la matriz $\mathbf{K}$. Suponiendo que otras condiciones permanecen inalteradas, cuando esta capa en su lugar realiza estricta *convolución*, el núcleo aprendido $\mathbf{K}'$ será el mismo que $\mathbf{K}$ después de $\mathbf{K}'$ se voltea tanto horizontal como verticalmente. Es decir, cuando la capa convolucional realiza estricta *convolución* para la entrada en [Referencia fig_correlation](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html#fig-correlation) y $\mathbf{K}'$, se obtendrá la misma salida en [Referencia fig_correlation](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html#fig-correlation) (correlación cruzada de la entrada y $\mathbf{K}$).

De acuerdo con la terminología estándar en la literatura de aprendizaje profundo, seguiremos refiriéndose a la operación de correlación cruzada como una convolución aunque, estrictamente hablando, es ligeramente diferente. Además, utilizamos el término *elemento* para referirnos a una entrada (o componente) de cualquier tensor que represente una representación de capa o un núcleo de convolución.

## Mapa de características y campo receptivo
Como se describe en [Referencia subsec_why-conv-channels](https://d2l.ai/chapter_convolutional-neural-networks/why-conv.html#subsec-why-conv-channels), la salida de la capa convolucional en
[Referencia fig_correlation](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html#fig-correlation)
En CNNs, para cualquier elemento $x$ de alguna capa, su *campo receptivo* se refiere a todos los elementos (de todas las capas anteriores) que pueden afectar al cálculo de $x$ durante la propagación hacia delante. Tenga en cuenta que el campo receptivo puede ser mayor que el tamaño real de la entrada.

Sigamos usando [Referencia fig_correlation](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html#fig-correlation) para explicar el campo receptivo. Dado el núcleo de convolución $2 \times 2$, el campo receptivo del elemento de salida sombreado (de valor $19$) son los cuatro elementos en la porción sombreada de la entrada. Ahora denotemos la salida $2 \times 2$ como $\mathbf{Y}$ y consideremos una CNN más profunda con una capa convolutiva $2 \times 2$ adicional que tome $\mathbf{Y}$ como su entrada, produciendo un único elemento $z$. En este caso, el campo receptivo de $z$ en $\mathbf{Y}$ incluye todos los cuatro elementos de $\mathbf{Y}$, mientras que el campo receptivo de la entrada incluye todos los nueve elementos de entrada. Así, cuando cualquier elemento de un mapa de características necesita un campo receptivo más grande para detectar características de entrada sobre un área más amplia, podemos construir una red más profunda.

Los campos receptivos derivan su nombre de la neurofisiología. Una serie de experimentos en una gama de animales utilizando diferentes estímulos
[Hubel.Wiesel.1959,Hubel.Wiesel.1962,Hubel.Wiesel.1968](https://d2l.ai/chapter_references/zreferences.html) estudiaron la respuesta del denominado sistema visual
Cortex en dichos estímulos. En general encontraron que los niveles inferiores responden a los bordes y formas relacionadas. Más adelante, [Field.1987](https://d2l.ai/chapter_references/zreferences.html) ilustró este efecto en las imágenes naturales con, lo que sólo se puede llamar, núcleos convolucionales. Reimprimimos una figura clave en [Referencia field_visual](https://d2l.ai/chapter_convolutional-neural-networks/conv-layer.html#field-visual) para ilustrar las sorprendentes similitudes.

![Figura y leyenda tomadas de [Field.1987](https://d2l.ai/chapter_references/zreferences.html): An example of coding with six different channels. (Left) Examples of the six types of sensor associated with each channel. (Right) Convolution of the image in (Middle) with the six sensors shown in (Left). The response of the individual sensors is determined by sampling these filtered images at a distance proportional to the size of the sensor (shown with dots). This diagram shows the response of only the even symmetric sensors.](../recursos/originales/field-visual.png)
<a id="field_visual"></a>

Como resulta, esta relación incluso se mantiene para las características calculadas por capas más profundas de redes entrenadas en tareas de clasificación de imágenes, como se demuestra, por ejemplo, en [Kuzovkin.Vicente.Petton.ea.2018](https://d2l.ai/chapter_references/zreferences.html). Baste decir, las convoluciones han demostrado ser una herramienta increíblemente poderosa para la visión informática, tanto en biología como en código. Como tal, no es sorprendente (en retrospectiva) que hayan anunciado el éxito reciente en el aprendizaje profundo.

## Resumen
El cálculo del núcleo requerido para una capa convolucional es una operación de correlación cruzada. Vimos que un simple anidado for-loop es todo lo que se requiere para calcular su valor. Si tenemos múltiples canales de entrada y múltiples canales de salida, estamos realizando una matriz--operación de matriz entre canales. Como se puede ver, el cálculo es sencillo y, lo más importante, altamente *local*. Esto permite una optimización de hardware significativa y muchos resultados recientes en la visión del ordenador sólo son posibles debido a eso. Después de todo, significa que los diseñadores de chips pueden invertir en computación rápida en lugar de memoria cuando se trata de optimizar para las convoluciones.

En términos de convoluciones en sí mismas, se pueden utilizar para muchos propósitos, por ejemplo, detectar bordes y líneas, difuminar imágenes, o afilarlas. Lo más importante, no es necesario que el estadístico (o ingeniero) inventa filtros adecuados. En su lugar, podemos simplemente *aprender* de datos. Esto reemplaza la heurística de ingeniería de características por estadísticas basadas en la evidencia. Por último, y muy agradablemente, estos filtros no sólo son ventajosos para construir redes profundas, sino que también corresponden a campos receptivos y mapas de características en el cerebro. Esto nos da confianza de que estamos en el camino correcto.

## Ejercicios
1. Construir una imagen `X` con bordes diagonales.
    1. ¿Qué sucede si aplica el kernel `K` en esta sección?
    1. ¿Qué pasa si transpones `X`?
    1. ¿Qué pasa si transpones `K`?
1. Diseñar algunos núcleos manualmente.
    1. Dado un vector direccional $\mathbf{v} = (v_1, v_2)$, derivar un núcleo de detección de bordes que detecta bordes ortogonales a $\mathbf{v}$, es decir, bordes en la dirección $(v_2, -v_1)$.
    1. Derive un operador de diferencia finita para la segunda derivada. ¿Cuál es el tamaño mínimo del núcleo convolucional asociado con ella? ¿Qué estructuras en imágenes responden más fuertemente a ella?
    1. ¿Cómo diseñaría un núcleo de desenfoque? ¿Por qué podría querer utilizar tal núcleo?
    1. ¿Cuál es el tamaño mínimo de un núcleo para obtener una derivada del orden $d$?
1. Cuando intentas encontrar automáticamente el gradiente para la clase `Conv2D` que creamos, ¿qué tipo de mensaje de error ves?
1. ¿Cómo representa una operación de correlación cruzada como multiplicación de matriz cambiando los tensores de entrada y núcleo?


[Debate del original](https://discuss.d2l.ai/t/66)
